In [106]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
sf = F = f
from pyspark.sql.window import Window
w = wd = Window

In [2]:
spark = SparkSession.builder.appName("Spark-SQL-Demo").getOrCreate()

In [3]:
df = df_employee = spark.read.csv("employee-dataset/employee-data.csv", header=True, inferSchema=True)
df.show()

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
|  Kelly| 26|  5100|        HR|
|   Liam| 31|  5900|        IT|
|    Mia| 34|  6800|   Finance|
|   Noah| 27|  5400|     Sales|
| Olivia| 36|  7100|        IT|
|  Peter| 29|  5600|        HR|
|  Quinn| 37|  7300|   Finance|
|   Rose| 28|  5500|     Sales|
|    Sam| 33|  6500|        IT|
|   Tina| 35|  7000|   Finance|
+-------+---+------+----------+
only showing top 20 rows



In [4]:
df.createOrReplaceTempView("employees")  # create temporary virtual table - no materialized table

In [26]:
result = spark.sql("SELECT * FROM employees WHERE age > 25 ORDER BY age")
result.show(20)

+------+---+------+----------+
|  name|age|salary|department|
+------+---+------+----------+
| Kelly| 26|  5100|        HR|
|   Uma| 26|  5200|        HR|
|Hannah| 26|  5100|        HR|
|  Tara| 26|  5100|        HR|
| Grace| 27|  5300|        HR|
|  Noah| 27|  5400|     Sales|
|Xavier| 27|  5300|     Sales|
| Fiona| 27|  5300|     Sales|
|Rachel| 27|  5300|     Sales|
| David| 28|  5500|     Sales|
|  Rose| 28|  5500|     Sales|
| Aaron| 28|  5500|   Finance|
| Mason| 28|  5500|   Finance|
| Daisy| 29|  5600|        HR|
| Peter| 29|  5600|        HR|
| Paula| 29|  5600|        HR|
|   Ivy| 29|  5700|     Sales|
|   Bob| 30|  6000|        IT|
|  Yara| 30|  6000|        HR|
| Kevin| 30|  6000|        IT|
+------+---+------+----------+
only showing top 20 rows



In [23]:
result = spark.sql("SELECT * FROM employees WHERE age < 26")
result.show()

+-----+---+------+----------+
| name|age|salary|department|
+-----+---+------+----------+
|Alice| 25|  5000|        HR|
+-----+---+------+----------+



In [8]:
result = spark.sql("SELECT department, ROUND(AVG(salary), 2) as avg_sal  FROM employees  GROUP BY department")
result.show()

+-------+
|avg_sal|
+-------+
|5909.09|
|5490.91|
|6646.15|
|6507.14|
+-------+



In [7]:
df.groupby("department").agg(f.avg("salary").alias("avg_salary")).show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|     Sales|5909.090909090909|
|        HR|5490.909090909091|
|   Finance|6646.153846153846|
|        IT|6507.142857142857|
+----------+-----------------+



## Exercise 1

Mention the details of department and their average salary by taking into consideration 
* all the employees whose age > 25
* and departments with an avg salary > 6000

In [18]:
result = spark.sql(
    """
    SELECT department, ROUND(AVG(salary), 2) as avg_salary  
    FROM employees
    WHERE age > 25
    GROUP BY department
    HAVING avg_salary > 6000
    """)
result.show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|   Finance|   6646.15|
|        IT|   6507.14|
+----------+----------+



In [22]:
result = (
    df
    .filter(f.col("age") > 25)
    .groupby("department")
    .agg(f.round(f.avg("salary"), 2).alias("avg_salary"))
    .filter(f.col("avg_salary") > 6000)
)
result.show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|   Finance|   6646.15|
|        IT|   6507.14|
+----------+----------+



## Exercise 2

Find all the pairs of employees who have same age but different names and display their names as name_a and name_b.

Bonus: remove reverse duplicates

In [121]:
result = spark.sql("""
    SELECT a.name as name_a, b.name as name_b
         , a.age as age
    FROM employees  a
    INNER JOIN employees  b
        ON a.age = b.age
        AND  a.name != b.name
    """)
print("Filtered on 'Bob' -> pairs are duplicated in reverse order:")
result.filter((f.col("name_a") == "Bob") | (f.col("name_b") == "Bob")).show()
print("All results by 'age':")
result.orderBy("age").show(10)
df.count(), result.count()

Filtered on 'Bob' -> pairs are duplicated in reverse order:
+------+------+---+
|name_a|name_b|age|
+------+------+---+
|   Bob|Walter| 30|
|   Bob| Kevin| 30|
|   Bob|  Yara| 30|
|  Yara|   Bob| 30|
| Kevin|   Bob| 30|
|Walter|   Bob| 30|
+------+------+---+

All results by 'age':
+------+------+---+
|name_a|name_b|age|
+------+------+---+
| Kelly|  Tara| 26|
| Kelly|Hannah| 26|
| Kelly|   Uma| 26|
|   Uma|  Tara| 26|
|   Uma|Hannah| 26|
|   Uma| Kelly| 26|
|Hannah|  Tara| 26|
|Hannah|   Uma| 26|
|Hannah| Kelly| 26|
|  Tara|Hannah| 26|
+------+------+---+
only showing top 10 rows



(49, 128)

In [95]:
result = (
    df.alias("a")
    .join(
        df.alias("b"), how="inner",
        on=(
            (f.col("a.age") == f.col("b.age"))
            & (f.col("a.name") != f.col("b.name"))
    ))
    .select(
        f.col("a.name").alias("name_a"),
        f.col("b.name").alias("name_b"),
        f.col("a.age").alias("age"),
    )
)
    
print("Filtered on 'Bob' -> pairs are duplicated in reverse order:")
result.filter((f.col("name_a") == "Bob") | (f.col("name_b") == "Bob")).show()
print("All results by 'age':")
result.orderBy("age").show(10)

Filtered on 'Bob' -> pairs are duplicated in reverse order:
+------+------+---+
|name_a|name_b|age|
+------+------+---+
|   Bob|Walter| 30|
|   Bob| Kevin| 30|
|   Bob|  Yara| 30|
|  Yara|   Bob| 30|
| Kevin|   Bob| 30|
|Walter|   Bob| 30|
+------+------+---+

All results by 'age':
+------+------+---+
|name_a|name_b|age|
+------+------+---+
| Kelly|  Tara| 26|
| Kelly|Hannah| 26|
| Kelly|   Uma| 26|
|   Uma|  Tara| 26|
|   Uma|Hannah| 26|
|   Uma| Kelly| 26|
|Hannah|  Tara| 26|
|Hannah|   Uma| 26|
|Hannah| Kelly| 26|
|  Tara|Hannah| 26|
+------+------+---+
only showing top 10 rows



### Bonus: cleaning reverse pair duplication

In [108]:
result = spark.sql("""
    SELECT *
        , ROW_NUMBER() OVER(PARTITION BY age ORDER BY name) as rn
    FROM employees
    WHERE age IN (29,30,31)
    """)
result.show(100)

+------+---+------+----------+---+
|  name|age|salary|department| rn|
+------+---+------+----------+---+
| Daisy| 29|  5600|        HR|  1|
|   Ivy| 29|  5700|     Sales|  2|
| Paula| 29|  5600|        HR|  3|
| Peter| 29|  5600|        HR|  4|
|   Bob| 30|  6000|        IT|  1|
| Kevin| 30|  6000|        IT|  2|
|Walter| 30|  6000|        IT|  3|
|  Yara| 30|  6000|        HR|  4|
| Bella| 31|  5900|     Sales|  1|
|  Liam| 31|  5900|        IT|  2|
|  Nora| 31|  5900|     Sales|  3|
+------+---+------+----------+---+



In [93]:
result = spark.sql("""
WITH empl_mod AS (
    SELECT *
        , ROW_NUMBER() OVER(PARTITION BY age ORDER BY name) as rn
    FROM employees
)
    SELECT a.name as name_a, b.name as name_b
        , a.age as age
    FROM empl_mod AS a
    INNER JOIN empl_mod AS b
        ON a.age = b.age
        AND  a.name != b.name
    WHERE a.rn == 1
    """)
print("Filtered on 'Bob' -> each name pair appears only once:")
result.filter((f.col("name_a") == "Bob") | (f.col("name_b") == "Bob")).show()
print("All results by 'age':")
result.orderBy("age").show(10)

Filtered on 'Bob' -> each name pair appears only once:
+------+------+---+
|name_a|name_b|age|
+------+------+---+
|   Bob|  Yara| 30|
|   Bob| Kevin| 30|
|   Bob|Walter| 30|
+------+------+---+

All results by 'age':
+------+------+---+
|name_a|name_b|age|
+------+------+---+
|Hannah| Kelly| 26|
|Hannah|   Uma| 26|
|Hannah|  Tara| 26|
| Fiona| Grace| 27|
| Fiona|  Noah| 27|
| Fiona|Xavier| 27|
| Fiona|Rachel| 27|
| Aaron| David| 28|
| Aaron| Mason| 28|
| Aaron|  Rose| 28|
+------+------+---+
only showing top 10 rows



In [109]:
# using withColumn to add row number column
result = (
    df
    .withColumn(
        "rn",
        (
            f.row_number().over(
                w.partitionBy(f.col("age"))
                .orderBy(f.col("name"))
             )
        )
    ).select(
        f.col("age"),
        f.col("name"),
        f.col("rn"),
    )
)
result.orderBy(f.col("age")).show(5)

+---+------+---+
|age|  name| rn|
+---+------+---+
| 25| Alice|  1|
| 26|Hannah|  1|
| 26| Kelly|  2|
| 26|  Tara|  3|
| 26|   Uma|  4|
+---+------+---+
only showing top 5 rows



In [112]:
# adding row number column in a select
result = (
    df
    .select(
        f.col("name"),
        f.col("age"),
        (
            f.row_number().over(
                w.partitionBy(f.col("age"))
                .orderBy(f.col("name"))
            )
        ).alias("rn"),
    )
)
result.orderBy(f.col("age")).show(5)

+------+---+---+
|  name|age| rn|
+------+---+---+
| Alice| 25|  1|
|Hannah| 26|  1|
| Kelly| 26|  2|
|  Tara| 26|  3|
|   Uma| 26|  4|
+------+---+---+
only showing top 5 rows



In [122]:
# adding row number as new column in new DataFrame and continue with that 
# (equivalent to a WITH AS sql statement)
df_w_col = (
    df
    .withColumn("rn", f.row_number().over(
        w.partitionBy(f.col("age"))
        .orderBy(f.col("name"))
     )))
result = (
    df_w_col.alias("a")
    .join(
        df_w_col.alias("b"), how="inner",
        on=(
            (f.col("a.age") == f.col("b.age"))
            & (f.col("a.name") != f.col("b.name"))
    ))
    .filter(f.col("a.rn") == 1)
    .select(
        f.col("a.name").alias("name_a"),
        f.col("b.name").alias("name_b"),
        f.col("a.age").alias("age"),
        f.col("a.rn").alias("rn_a"),
        f.col("b.rn").alias("rn_b"),
    )
)
    
print("Filtered on 'Bob' -> each name pair appears only once:")
result.filter((f.col("name_a") == "Bob") | (f.col("name_b") == "Bob")).show()
print("All results by 'age':")
result.orderBy("age").show(10)
df.count(), result.count()

Filtered on 'Bob' -> each name pair appears only once:
+------+------+---+----+----+
|name_a|name_b|age|rn_a|rn_b|
+------+------+---+----+----+
|   Bob| Kevin| 30|   1|   2|
|   Bob|Walter| 30|   1|   3|
|   Bob|  Yara| 30|   1|   4|
+------+------+---+----+----+

All results by 'age':
+------+------+---+----+----+
|name_a|name_b|age|rn_a|rn_b|
+------+------+---+----+----+
|Hannah| Kelly| 26|   1|   2|
|Hannah|  Tara| 26|   1|   3|
|Hannah|   Uma| 26|   1|   4|
| Fiona| Grace| 27|   1|   2|
| Fiona|  Noah| 27|   1|   3|
| Fiona|Rachel| 27|   1|   4|
| Fiona|Xavier| 27|   1|   5|
| Aaron| David| 28|   1|   2|
| Aaron| Mason| 28|   1|   3|
| Aaron|  Rose| 28|   1|   4|
+------+------+---+----+----+
only showing top 10 rows



(49, 33)

## Exercise 3

Use rank() function to assign a rank to employees based on their salary within each department, with the highest salary getting first rank

In [129]:
spark.sql("""
SELECT 
    *,
    RANK() OVER (PARTITION BY department  ORDER BY salary DESC) AS salary_rank
FROM employees e
""").show()

+-------+---+------+----------+-----------+
|   name|age|salary|department|salary_rank|
+-------+---+------+----------+-----------+
|  Frank| 40|  7500|   Finance|          1|
|  Wendy| 39|  7400|   Finance|          2|
|  Quinn| 37|  7300|   Finance|          3|
|   Jack| 38|  7200|   Finance|          4|
|Charlie| 35|  7000|   Finance|          5|
|   Tina| 35|  7000|   Finance|          5|
|    Mia| 34|  6800|   Finance|          7|
|  Ethan| 33|  6400|   Finance|          8|
| Quincy| 33|  6400|   Finance|          8|
|  Isaac| 32|  6200|   Finance|         10|
|Ulysses| 32|  6200|   Finance|         10|
|  Aaron| 28|  5500|   Finance|         12|
|  Mason| 28|  5500|   Finance|         12|
|  Laura| 34|  6800|        HR|          1|
|   Yara| 30|  6000|        HR|          2|
|  Peter| 29|  5600|        HR|          3|
|  Daisy| 29|  5600|        HR|          3|
|  Paula| 29|  5600|        HR|          3|
|  Grace| 27|  5300|        HR|          6|
|    Uma| 26|  5200|        HR| 

In [139]:
result = (
    df.withColumn("salary_rank", f.rank().over(
        Window
        .partitionBy(f.col("department"))
        .orderBy(f.col("salary").desc())
    ))
)
result.show()

+-------+---+------+----------+-----------+
|   name|age|salary|department|salary_rank|
+-------+---+------+----------+-----------+
|  Frank| 40|  7500|   Finance|          1|
|  Wendy| 39|  7400|   Finance|          2|
|  Quinn| 37|  7300|   Finance|          3|
|   Jack| 38|  7200|   Finance|          4|
|Charlie| 35|  7000|   Finance|          5|
|   Tina| 35|  7000|   Finance|          5|
|    Mia| 34|  6800|   Finance|          7|
|  Ethan| 33|  6400|   Finance|          8|
| Quincy| 33|  6400|   Finance|          8|
|  Isaac| 32|  6200|   Finance|         10|
|Ulysses| 32|  6200|   Finance|         10|
|  Aaron| 28|  5500|   Finance|         12|
|  Mason| 28|  5500|   Finance|         12|
|  Laura| 34|  6800|        HR|          1|
|   Yara| 30|  6000|        HR|          2|
|  Peter| 29|  5600|        HR|          3|
|  Daisy| 29|  5600|        HR|          3|
|  Paula| 29|  5600|        HR|          3|
|  Grace| 27|  5300|        HR|          6|
|    Uma| 26|  5200|        HR| 